# 05 — Model 2: PSM (Digital Twins)

1:1 nearest-neighbor matching on propensity score with a caliper.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

df = pd.read_csv(Path("processed/df_analysis.csv"))
df["internet_usage"] = pd.Categorical(df["internet_usage"], categories=["Low", "Medium", "High", "Extreme"], ordered=True)
for col in ["tv_product", "mobile_product", "commune"]:
    df[col] = df[col].astype("category")

model_cols = [
    "churned", "has_booster", "age", "tenure",
    "internet_usage", "tv_product", "mobile_product", "commune",
]
df_clean = df[model_cols].dropna().copy()
print(f"Rows for PSM: {len(df_clean):,}")

In [ ]:
ps_formula = (
    "has_booster ~ age + tenure + C(internet_usage) + "
    "C(tv_product) + C(mobile_product) + C(commune)"
)
ps_model = smf.logit(formula=ps_formula, data=df_clean).fit(disp=False)
df_clean["propensity_score"] = ps_model.predict(df_clean)

treated = df_clean[df_clean["has_booster"] == 1].copy().reset_index(drop=True)
control = df_clean[df_clean["has_booster"] == 0].copy()

nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(control[["propensity_score"]])
distances, idx = nn.kneighbors(treated[["propensity_score"]])

matched_controls = control.iloc[idx.flatten()].reset_index(drop=True)
treated["match_distance"] = distances.flatten()

caliper = 0.02
keep = treated["match_distance"] <= caliper

matched = pd.DataFrame({
    "treated_churn": treated.loc[keep, "churned"].to_numpy(),
    "control_churn": matched_controls.loc[keep, "churned"].to_numpy(),
    "internet_usage": treated.loc[keep, "internet_usage"].astype(str).to_numpy(),
    "match_distance": treated.loc[keep, "match_distance"].to_numpy(),
})
matched["pair_effect"] = matched["treated_churn"] - matched["control_churn"]

att_overall = float(matched["pair_effect"].mean())
att_by_tier = (
    matched.groupby("internet_usage", as_index=False)["pair_effect"]
           .agg(att="mean", n_pairs="count")
           .sort_values("internet_usage")
)

print(f"Treated customers: {len(treated):,}")
print(f"Matched pairs within caliper {caliper:.3f}: {len(matched):,} ({len(matched)/len(treated):.1%} coverage)")
print(f"Digital Twins ATT overall: {att_overall*100:+.2f} pp")
display(att_by_tier.assign(att_pp=lambda d: d["att"] * 100))